In [1]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from scipy.ndimage import convolve

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TEMPLATE_PROCESSED_DIR = PROCESSED_DIR / "template_images"
TEMPLATE_BINARY_DIR = TEMPLATE_PROCESSED_DIR / "binary"
TEMPLATE_SKELETON_DIR = TEMPLATE_PROCESSED_DIR / "skeleton"

STUDENT_FEATURES_PATH = PROCESSED_DIR / "image_features.csv"

In [2]:
def count_skeleton_points(skeleton_img):
    skel = (skeleton_img > 0).astype(np.uint8)

    kernel = np.array([
        [1, 1, 1],
        [1, 0, 1],
        [1, 1, 1]
    ], dtype=np.uint8)

    neighbor_count = convolve(skel, kernel, mode="constant", cval=0)

    endpoints = np.logical_and(skel == 1, neighbor_count == 1).sum()
    intersections = np.logical_and(skel == 1, neighbor_count >= 3).sum()

    return int(endpoints), int(intersections)

In [3]:
def extract_template_features(template_id):
    binary_path = TEMPLATE_BINARY_DIR / f"template_{template_id:02d}_binary.png"
    skeleton_path = TEMPLATE_SKELETON_DIR / f"template_{template_id:02d}_skeleton.png"

    binary = cv2.imread(str(binary_path), cv2.IMREAD_GRAYSCALE)
    skeleton = cv2.imread(str(skeleton_path), cv2.IMREAD_GRAYSCALE)

    if binary is None or skeleton is None:
        raise ValueError(f"Missing template image for template {template_id}")

    h, w = binary.shape
    image_area = h * w

    ink_pixels = int(np.count_nonzero(binary))
    ink_density = ink_pixels / image_area

    skeleton_length = int(np.count_nonzero(skeleton))
    skeleton_density = skeleton_length / image_area

    endpoints_count, intersections_count = count_skeleton_points(skeleton)

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary,
        connectivity=8
    )

    component_areas = stats[1:, cv2.CC_STAT_AREA]
    valid_components = component_areas[component_areas > 20]
    connected_components_count = int(len(valid_components))

    contours, _ = cv2.findContours(
        binary,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    valid_contours = [
        cnt for cnt in contours
        if cv2.contourArea(cnt) > 20
    ]

    contour_count = len(valid_contours)

    ys, xs = np.where(binary > 0)

    if len(xs) > 0 and len(ys) > 0:
        x_min, x_max = xs.min(), xs.max()
        y_min, y_max = ys.min(), ys.max()

        bbox_width = int(x_max - x_min + 1)
        bbox_height = int(y_max - y_min + 1)
        bbox_area = bbox_width * bbox_height
        bbox_area_ratio = bbox_area / image_area
        bbox_aspect_ratio = bbox_width / (bbox_height + 1e-6)
    else:
        bbox_width = 0
        bbox_height = 0
        bbox_area_ratio = 0
        bbox_aspect_ratio = 0

    return {
        "template_id": template_id,
        "ink_pixels": ink_pixels,
        "ink_density": ink_density,
        "skeleton_length": skeleton_length,
        "skeleton_density": skeleton_density,
        "endpoints_count": endpoints_count,
        "intersections_count": intersections_count,
        "connected_components_count": connected_components_count,
        "contour_count": contour_count,
        "bbox_width": bbox_width,
        "bbox_height": bbox_height,
        "bbox_area_ratio": bbox_area_ratio,
        "bbox_aspect_ratio": bbox_aspect_ratio
    }

In [5]:
template_features_list = []

for template_id in range(1, 10):
    features = extract_template_features(template_id)
    template_features_list.append(features)

template_features = pd.DataFrame(template_features_list)

print(template_features.shape)
template_features

(9, 13)


,template_id,ink_pixels,ink_density,skeleton_length,skeleton_density,endpoints_count,intersections_count,connected_components_count,contour_count,bbox_width,bbox_height,bbox_area_ratio,bbox_aspect_ratio
0,1,12807,0.010243,1535,0.001228,15,17,5,3,889,722,0.513341,1.231302
1,2,1970,0.001480,50,0.000038,24,2,12,12,1239,219,0.203883,5.657534
2,3,10405,0.009543,1669,0.001531,23,1,34,33,1202,361,0.397957,3.329640
3,4,2080,0.002412,64,0.000074,30,3,17,17,411,243,0.115807,1.691358
4,5,1077,0.009066,330,0.002778,4,7,1,1,133,136,0.152256,0.977941
5,6,2907,0.002317,139,0.000111,71,4,29,29,323,355,0.091378,0.909859
6,7,13856,0.011142,2008,0.001615,27,8,5,5,1184,636,0.605526,1.861635
7,8,12725,0.016499,1601,0.002076,89,15,7,1,342,397,0.176040,0.861461
8,9,12795,0.010244,1915,0.001533,7,20,1,1,916,159,0.116609,5.761006


In [6]:
template_features.to_csv(
    PROCESSED_DIR / "template_features.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved template_features.csv")

Saved template_features.csv


In [8]:
reference_summary = {
    "ref_total_ink_pixels": template_features["ink_pixels"].sum(),
    "ref_total_skeleton_length": template_features["skeleton_length"].sum(),
    "ref_total_endpoints": template_features["endpoints_count"].sum(),
    "ref_total_intersections": template_features["intersections_count"].sum(),
    "ref_total_components": template_features["connected_components_count"].sum(),
    "ref_total_contours": template_features["contour_count"].sum(),

    "ref_mean_ink_density": template_features["ink_density"].mean(),
    "ref_mean_skeleton_density": template_features["skeleton_density"].mean(),
    "ref_mean_bbox_area_ratio": template_features["bbox_area_ratio"].mean(),
    "ref_mean_bbox_aspect_ratio": template_features["bbox_aspect_ratio"].mean()
}

reference_summary

{'ref_total_ink_pixels': np.int64(70622),
 'ref_total_skeleton_length': np.int64(9311),
 'ref_total_endpoints': np.int64(290),
 'ref_total_intersections': np.int64(77),
 'ref_total_components': np.int64(111),
 'ref_total_contours': np.int64(102),
 'ref_mean_ink_density': np.float64(0.008104961401883992),
 'ref_mean_skeleton_density': np.float64(0.001220262785363044),
 'ref_mean_bbox_area_ratio': np.float64(0.26364412955391225),
 'ref_mean_bbox_aspect_ratio': np.float64(2.4757485336364917)}

In [9]:
image_features = pd.read_csv(STUDENT_FEATURES_PATH)

print(image_features.shape)
image_features.head()

(89, 31)


,case_id,image_width,image_height,is_landscape,ink_pixels,ink_density,stroke_darkness_mean,stroke_darkness_std,skeleton_length,skeleton_density,...,components_per_ink_area,bbox_width,bbox_height,bbox_area_ratio,bbox_aspect_ratio,bbox_center_x_norm,bbox_center_y_norm,drawing_center_distance_norm,perseveration_score,distortion_proxy_score
0,1,1241,1755,0,27946,0.012831,63.727796,49.882035,7015,0.003221,...,0.004723,1200,1687,0.929496,0.711322,0.493554,0.511681,0.010238,-1.366521,-0.303671
1,2,1241,1755,0,33761,0.015501,117.697965,63.608772,6718,0.003085,...,0.004206,1183,1643,0.892428,0.720024,0.500403,0.478632,0.017448,0.074930,0.471576
2,3,1241,1755,0,29999,0.013774,81.395313,53.612199,7692,0.003532,...,0.004067,1165,1494,0.799149,0.779786,0.478646,0.433903,0.055358,-1.116456,-0.379011
3,4,1755,1241,1,31808,0.014605,94.960670,47.822996,7157,0.003286,...,0.004464,1512,1155,0.801835,1.309091,0.511396,0.489122,0.011226,-0.597039,-0.231889
4,5,1241,1755,0,33813,0.015525,125.515246,64.123418,7413,0.003404,...,0.004081,1191,1571,0.859091,0.758116,0.497180,0.479202,0.017059,-0.059099,-0.291300


In [10]:
template_deviation_features = pd.DataFrame()

template_deviation_features["case_id"] = image_features["case_id"]

template_deviation_features["skeleton_length_deviation"] = abs(
    image_features["skeleton_length"] - reference_summary["ref_total_skeleton_length"]
) / (reference_summary["ref_total_skeleton_length"] + 1e-6)

template_deviation_features["endpoints_deviation"] = abs(
    image_features["endpoints_count"] - reference_summary["ref_total_endpoints"]
) / (reference_summary["ref_total_endpoints"] + 1e-6)

template_deviation_features["intersections_deviation"] = abs(
    image_features["intersections_count"] - reference_summary["ref_total_intersections"]
) / (reference_summary["ref_total_intersections"] + 1e-6)

template_deviation_features["components_deviation"] = abs(
    image_features["connected_components_count"] - reference_summary["ref_total_components"]
) / (reference_summary["ref_total_components"] + 1e-6)

template_deviation_features["contours_deviation"] = abs(
    image_features["contour_count"] - reference_summary["ref_total_contours"]
) / (reference_summary["ref_total_contours"] + 1e-6)

template_deviation_features["ink_density_deviation"] = abs(
    image_features["ink_density"] - reference_summary["ref_mean_ink_density"]
) / (reference_summary["ref_mean_ink_density"] + 1e-6)

template_deviation_features["skeleton_density_deviation"] = abs(
    image_features["skeleton_density"] - reference_summary["ref_mean_skeleton_density"]
) / (reference_summary["ref_mean_skeleton_density"] + 1e-6)

In [11]:
template_deviation_features["template_deviation_score"] = (
    template_deviation_features["skeleton_length_deviation"] +
    template_deviation_features["endpoints_deviation"] +
    template_deviation_features["intersections_deviation"] +
    template_deviation_features["components_deviation"] +
    template_deviation_features["contours_deviation"] +
    template_deviation_features["ink_density_deviation"] +
    template_deviation_features["skeleton_density_deviation"]
) / 7

template_deviation_features.head()

,case_id,skeleton_length_deviation,endpoints_deviation,intersections_deviation,components_deviation,contours_deviation,ink_density_deviation,skeleton_density_deviation,template_deviation_score
0,1,0.246590,0.296552,0.233766,0.189189,0.039216,0.583070,1.638180,0.460938
1,2,0.278488,0.475862,3.701299,0.279279,0.117647,0.912449,1.526520,1.041649
2,3,0.173880,0.434483,2.181818,0.099099,0.019608,0.699358,1.892705,0.785850
3,4,0.231339,0.675862,1.363636,0.279279,0.088235,0.801825,1.691567,0.733106
4,5,0.203845,0.382759,1.038961,0.243243,0.137255,0.915395,1.787812,0.672753


In [12]:
output_path = PROCESSED_DIR / "template_deviation_features.csv"

template_deviation_features.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", output_path)
print("Shape:", template_deviation_features.shape)

Saved: d:\FCAI\Year 3\2nd semster\Cognitive\bender-gestalt-project\data\processed\template_deviation_features.csv
Shape: (89, 9)
